# CropCare: Phases 1 to 4 on Colab

This notebook runs the CropCare build plan from data to a phone-ready model:

1. **Data**: download PlantVillage and PlantDoc, add your field photos, build the manifest, check for leakage
2. **Baseline**: train on PlantVillage only and test on all three sets
3. **Close the gap**: one technique at a time, compared by PlantDoc test macro F1
4. **Convert**: TFLite files checked on the phone's CPU path, with size and accuracy

**Before you start**
- Runtime > Change runtime type > **T4 GPU**.
- Add two Colab secrets (key icon on the left): `KAGGLE_USERNAME` and `KAGGLE_KEY`, from kaggle.com > Settings > API > Create New Token. Turn on notebook access for both.
- Put your own photos in Google Drive as `MyDrive/cropcare/field.zip`, holding `images/` and `labels.csv` (columns `filename,label,labelled_by,date,place`). Optional: `MyDrive/cropcare/backgrounds.zip`, 20 or more photos of soil, hands and plants, for background swap.

**Rules this notebook enforces**
- Field photos are never used for training or tuning. They are tested on the baseline once and on the final model once.
- The unsure threshold is tuned on PlantDoc **val**, never on a test set.
- Trained models and every result file are saved to Drive, so a disconnect does not lose work. Re-running a cell skips work that is already saved.

## Settings

In [ ]:
REPO_URL = ""            # your GitHub repo, e.g. "https://github.com/<you>/cropcare.git"
DRIVE_DIR = "/content/drive/MyDrive/cropcare"   # results and models are kept here
MODEL_VERSION = "1.0.0"  # written into the model metadata the app shows
THRESHOLD_TARGET = 0.9   # accuracy wanted on the photos the app does answer
MAX_ACCURACY_DROP = 0.02 # Phase 4: smallest .tflite that loses at most this much accuracy
QUICK_TEST = False       # True: 1 epoch per stage, to check the notebook end to end

## Setup

In [ ]:
import csv, json, os, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = Path("/content/cropcare")
    if not (REPO / "ml").is_dir():
        if not REPO_URL:
            raise SystemExit("Set REPO_URL in the settings cell, or upload the cropcare folder to /content/cropcare.")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
    ML = REPO / "ml"
    DATA = Path("/content/data")                 # local disk: much faster than Drive for training
    RECORDS = Path(DRIVE_DIR)
else:                                            # running outside Colab (used to test this notebook)
    ML = Path.cwd()
    DATA = Path(os.environ.get("CROPCARE_DATA", ML.parent / "data"))
    RECORDS = Path(os.environ.get("CROPCARE_RECORDS", DATA / "records"))

RUNS = RECORDS / "runs"
for d in (DATA, RECORDS, RUNS):
    d.mkdir(parents=True, exist_ok=True)
os.chdir(ML)

EPOCHS = ["--epochs-head", "1", "--epochs-finetune", "1"] if QUICK_TEST else []
N_BOOT = "200" if QUICK_TEST else "2000"


def run(script, *args):
    """Run one of the project's scripts and show its output as it happens."""
    cmd = [sys.executable, "-u", f"scripts/{script}", "--data-dir", str(DATA), *map(str, args)]
    print("$", " ".join(cmd[2:]))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        if " I tensorflow/" in line or " W tensorflow/" in line or line.startswith(("I0000", "W0000")):
            continue  # TensorFlow start-up chatter
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"{script} failed (exit {proc.returncode})")


def show_md(path):
    from IPython.display import Markdown, display
    display(Markdown(Path(path).read_text(encoding="utf-8")))


print("ML folder:", ML)
print("Data:     ", DATA)
print("Records:  ", RECORDS)

In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ai-edge-litert", "pillow"], check=True)
import tensorflow as tf
print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU") or "none")
major, minor = map(int, tf.__version__.split(".")[:2])
if not (2, 16) <= (major, minor) < (2, 21):
    print("Note: this project was tested with TensorFlow 2.16 to 2.20. Newer versions usually work; check the outputs.")
if IN_COLAB and not tf.config.list_physical_devices("GPU"):
    print("No GPU. Runtime > Change runtime type > T4 GPU, then run the notebook again.")

## Phase 1: Data

### 1.1 PlantVillage
Downloads the Kaggle copy (about 4.4 GB, a few minutes) and copies only the tomato, potato and maize folders, colour and segmented, into the data folder.

In [ ]:
PV = DATA / "raw/plantvillage"
PV_SEG = DATA / "raw/plantvillage_segmented"
CROPS = ("Tomato", "Potato", "Corn")

if PV.is_dir() and any(PV.iterdir()):
    print("PlantVillage already in place:", PV)
else:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    import kagglehub
    src = Path(kagglehub.dataset_download("abdallahalidev/plantvillage-dataset"))
    found = {}
    for kind in ("color", "segmented"):
        dirs = [p for p in src.rglob(kind) if p.is_dir() and any(c.name.startswith(CROPS) for c in p.iterdir())]
        if not dirs:
            raise SystemExit(f"Could not find the '{kind}' folder in {src}")
        found[kind] = dirs[0]
    for kind, dest in (("color", PV), ("segmented", PV_SEG)):
        for cls in sorted(found[kind].iterdir()):
            if cls.is_dir() and cls.name.startswith(CROPS):
                shutil.copytree(cls, dest / cls.name, dirs_exist_ok=True)
for d in (PV, PV_SEG):
    if d.is_dir():
        print(d.name, {c.name: sum(1 for _ in c.iterdir()) for c in sorted(d.iterdir())})

### 1.2 PlantDoc

In [ ]:
PD = DATA / "raw/plantdoc"
if not (PD / "test").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/pratikkayal/PlantDoc-Dataset", str(PD)], check=True)
print("PlantDoc train folders:", len(list((PD / "train").iterdir())), "| test folders:", len(list((PD / "test").iterdir())))

### 1.3 Your field photos and background photos
Field photos come from `field.zip` in your Drive folder. Without them the notebook still runs, but there is no honest final number.

In [ ]:
def unzip_from_drive(name, dest, marker=None):
    """Extract MyDrive/cropcare/<name> into dest once. Accepts a zip whose files sit inside one top folder."""
    z = RECORDS / name
    if not (dest.is_dir() and any(dest.iterdir())):
        if not z.is_file():
            return False
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        entries = list(dest.iterdir())
        if len(entries) == 1 and entries[0].is_dir() and (marker is None or not (dest / marker).exists()):
            top = entries[0]
            for item in list(top.iterdir()):
                shutil.move(str(item), str(dest / item.name))
            top.rmdir()
    return marker is None or (dest / marker).exists()

HAVE_FIELD = unzip_from_drive("field.zip", DATA / "field", marker="labels.csv")
HAVE_BACKGROUNDS = unzip_from_drive("backgrounds.zip", DATA / "backgrounds")
print("Field photos:", "yes" if HAVE_FIELD else "NO (add field.zip to Drive for the final test)")
print("Backgrounds: ", "yes" if HAVE_BACKGROUNDS else "no (background swap will be skipped)")

### 1.4 Manifest, unreadable files and data card

In [ ]:
run("prepare_data.py")
run("verify_images.py", "--drop")

### 1.5 Leakage check
Finds val/test photos that are near-duplicates of training photos. Look at the pairs below before dropping them.

In [ ]:
run("check_leakage.py")
import matplotlib.pyplot as plt
from PIL import Image
with open(DATA / "leakage_pairs.csv", newline="", encoding="utf-8") as f:
    pairs = list(csv.DictReader(f))
print(len(pairs), "pairs")
show = pairs[:6]
if show:
    fig, axes = plt.subplots(len(show), 2, figsize=(6, 3 * len(show)), squeeze=False)
    for row, p in zip(axes, show):
        for ax, key in zip(row, ("eval_path", "train_path")):
            ax.imshow(Image.open(DATA / p[key]).convert("RGB")); ax.axis("off")
            ax.set_title(f"{key.split('_')[0]} ({p['distance']} bits)", fontsize=9)
    plt.tight_layout(); plt.show()

In [ ]:
DROP_LEAKS = True   # set False if the pairs above are different photos
if DROP_LEAKS:
    run("check_leakage.py", "--drop-from-pairs")

In [ ]:
run("data_card.py", "--out", ML / "data_card.md")
for name in ("manifest.csv", "leakage_pairs.csv"):
    shutil.copy(DATA / name, RECORDS / name)
shutil.copy(ML / "data_card.md", RECORDS / "data_card.md")
show_md(ML / "data_card.md")

## Phase 2: Baseline (PlantVillage only)
Training takes a while on a T4. If Colab disconnects, run the notebook again: finished runs are loaded from Drive.

In [ ]:
def train(name, *args):
    out = RUNS / name
    if (out / "model.keras").is_file():
        print(f"{name}: already trained, using {out / 'model.keras'}")
    else:
        run("train.py", *args, *EPOCHS, "--out", out)
    return out / "model.keras"


def evaluate(model, source, split="test"):
    run("evaluate.py", "--model", model, "--source", source, "--split", split, "--n-boot", N_BOOT)
    suffix = "" if Path(model).suffix == ".keras" else f"_{Path(model).stem}"
    return json.loads((Path(model).parent / f"eval_{source}_{split}{suffix}.json").read_text(encoding="utf-8"))


BASELINE = train("p2_baseline", "--sources", "plantvillage")
baseline = {s: evaluate(BASELINE, s) for s in ("plantvillage", "plantdoc")}
if HAVE_FIELD:
    baseline["field"] = evaluate(BASELINE, "field")   # the plan's "before" number on field photos

## Phase 3: Close the lab-to-field gap
Each run adds one change. Background swap runs only if `backgrounds.zip` was found.

In [ ]:
if HAVE_BACKGROUNDS and PV_SEG.is_dir():
    with open(DATA / "manifest.csv", newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    if not any(r["source"] == "plantvillage_bgswap" for r in rows):
        run("bg_swap.py", "--segmented", PV_SEG, "--backgrounds", DATA / "backgrounds",
            *(["--per-class", "5"] if QUICK_TEST else []))
        run("data_card.py", "--out", ML / "data_card.md")
        shutil.copy(DATA / "manifest.csv", RECORDS / "manifest.csv")
    HAVE_BGSWAP = True
else:
    HAVE_BGSWAP = False
    print("Skipping background swap (no backgrounds or no segmented folder).")

MIX = ["--sources", "plantvillage", "plantdoc", "--weights", "0.6", "0.4"]
MIX_BG = ["--sources", "plantvillage", "plantdoc", "plantvillage_bgswap", "--weights", "0.45", "0.35", "0.2"]
EXPERIMENTS = {"p2_baseline": None, "p3_1_mixed": MIX, "p3_2_strong": [*MIX, "--augment", "strong"]}
if HAVE_BGSWAP:
    EXPERIMENTS["p3_3_bgswap"] = [*MIX_BG, "--augment", "strong"]
    EXPERIMENTS["p3_4_weights"] = [*MIX_BG, "--augment", "strong", "--class-weights"]
else:
    EXPERIMENTS["p3_4_weights"] = [*MIX, "--augment", "strong", "--class-weights"]

phase3 = {"p2_baseline": baseline["plantdoc"]}
for name, args in EXPERIMENTS.items():
    if args is None:
        continue
    phase3[name] = evaluate(train(name, *args), "plantdoc")

In [ ]:
import pandas as pd
table = pd.DataFrame([
    {"run": name, "macro F1": f"{r['macro_f1']:.3f}", "F1 95% CI": f"{r['macro_f1_ci95'][0]:.3f} to {r['macro_f1_ci95'][1]:.3f}",
     "accuracy": f"{r['accuracy']:.1%}", "images": r["n_images"]}
    for name, r in phase3.items()
])
display(table)
BEST_RUN = max(phase3, key=lambda n: phase3[n]["macro_f1"])
print("Best by PlantDoc test macro F1:", BEST_RUN)
print("Overlapping confidence intervals mean the difference may be chance; prefer the simpler run if so.")

### Unsure threshold (tuned on PlantDoc val)
Change `BEST_RUN` here if you prefer a different run from the table above.

In [ ]:
BEST = RUNS / BEST_RUN / "model.keras"
run("tune_threshold.py", "--model", BEST, "--target", 0.0 if QUICK_TEST else THRESHOLD_TARGET,
    *(["--grid-start", "0.05"] if QUICK_TEST else []))
threshold = json.loads((BEST.parent / "threshold.json").read_text(encoding="utf-8"))
print("threshold:", threshold["threshold"])

### Final test on field photos (once)
This cell records which model was tested. It refuses to test a different model afterwards, so the field number stays honest. Delete `final_field_test.json` in Drive only if you really start over.

In [ ]:
marker = RECORDS / "final_field_test.json"
if not HAVE_FIELD:
    print("No field photos: add field.zip to Drive, then run from section 1.3 again.")
else:
    if marker.is_file():
        done = json.loads(marker.read_text(encoding="utf-8"))
        if done["model"] != str(BEST):
            raise SystemExit(f"Field photos were already used for the final test of {done['model']} on {done['when']}.")
        print("Final field test already done for this model; showing it again.")
    final_field = evaluate(BEST, "field")
    if not marker.is_file():
        marker.write_text(json.dumps({"model": str(BEST), "when": datetime.now(timezone.utc).isoformat()}, indent=2))
    show_md(BEST.parent / "eval_field_test.md")

## Phase 4: Convert to TFLite
Checks every file with LiteRT's XNNPACK delegate (the phone's CPU path). Accuracy is measured on field photos, or on PlantDoc test if you have none yet.

In [ ]:
export = BEST.parent / "export"
if not (export / "conversion.json").is_file():
    run("convert.py", "--model", BEST, "--version", MODEL_VERSION, "--eval-source", "field" if HAVE_FIELD else "plantdoc",
        *(["--representative", "20"] if QUICK_TEST else []))
show_md(export / "conversion.md")

conv = json.loads((export / "conversion.json").read_text(encoding="utf-8"))
keras_acc = conv["rows"][0]["accuracy"]
ok = [r for r in conv["rows"][1:] if keras_acc - r["accuracy"] <= MAX_ACCURACY_DROP]
if not ok:
    raise SystemExit(f"No .tflite stays within {MAX_ACCURACY_DROP:.0%} of the Keras model; look at the table.")
chosen = min(ok, key=lambda r: r["size_bytes"])
variant = chosen["file"].removeprefix("cropcare_").removesuffix(".tflite")
app_model = RECORDS / "app_model"
app_model.mkdir(exist_ok=True)
shutil.copy(export / chosen["file"], app_model / "cropcare.tflite")
shutil.copy(export / f"cropcare_{variant}.meta.json", app_model / "model_meta.json")
print(f"Chosen: {chosen['name']} ({chosen['size_bytes'] / 1e6:.2f} MB, accuracy {chosen['accuracy']:.1%})")
print(f"Copy both files from {app_model} into app/assets/model/ in the repository.")

## Numbers for results.md
Paste these into `results.md`. Phone speed comes later, from the app's About screen.

In [ ]:
def acc(r):
    return f"{r['accuracy']:.1%} ({r['accuracy_ci95'][0]:.1%} to {r['accuracy_ci95'][1]:.1%})"

def f1(r):
    return f"{r['macro_f1']:.3f} ({r['macro_f1_ci95'][0]:.3f} to {r['macro_f1_ci95'][1]:.3f})"

lines = ["## Phase 2: baseline", "", "| Test set | Accuracy (95% CI) | Macro F1 (95% CI) |", "|---|---|---|"]
for s, label in (("plantvillage", "PlantVillage test"), ("plantdoc", "PlantDoc test"), ("field", "Field photos")):
    if s in baseline:
        r = baseline[s]
        lines.append(f"| {label} ({r['n_images']} images) | {acc(r)} | {f1(r)} |")
lines += ["", "## Phase 3", "", "| Run | PlantDoc test macro F1 (95% CI) | PlantDoc test accuracy |", "|---|---|---|"]
lines += [f"| {n} | {f1(r)} | {r['accuracy']:.1%} |" for n, r in phase3.items()]
t = threshold["threshold"]
row = next(x for x in threshold["table"] if abs(x["threshold"] - t) < 1e-9)
lines += ["", f"Unsure threshold: {t:.2f} (PlantDoc val, {threshold['n_val']} images). "
          f"Coverage {row['coverage']:.1%}, accuracy on answered photos {row['accuracy_answered']:.1%}."]
if HAVE_FIELD:
    lines += [f"Final model on field photos: accuracy {acc(final_field)}, macro F1 {f1(final_field)}."]
lines += ["", "## Phase 4", "", "| Version | Size (MB) | Accuracy |", "|---|---|---|"]
lines += [f"| {r['name']} | {r['size_bytes'] / 1e6:.2f} | {r['accuracy']:.1%} |" for r in conv["rows"]]
lines += ["", f"Chosen file: {chosen['file']}"]
text = "\n".join(lines)
(RECORDS / "results_snippet.md").write_text(text + "\n", encoding="utf-8")
print(text)